# Fake News Detection & Retrieval: Milestone 2 - Model Experiments & Ablations
**Course:** SPPU Information Retrieval (IR) Mini Project
**Models:** Multinomial Naive Bayes, Logistic Regression, Linear SVM (Calibrated)


## 1. Setup & Load Cleaned Data


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from ml.config import PROCESSED, MODELS, REPORTS, SEED, TEST_SIZE
from ml.engine import get_global_influential_terms, get_local_term_contributions

df = pd.read_parquet(PROCESSED / "cleaned_corpus.parquet")
print(f"Loaded {len(df)} documents.")


## 2. Train / Test Split & TF-IDF Vector Space Model
**Golden IR Rule:** Fit `TfidfVectorizer` **on train split only** to prevent test information leakage into IDF weights.


In [ ]:
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, stratify=df["label"], random_state=SEED)

vec = TfidfVectorizer(ngram_range=(1, 2), max_features=50_000, min_df=3, max_df=0.9, sublinear_tf=True, norm="l2")
X_tr = vec.fit_transform(train_df["clean"])
X_te = vec.transform(test_df["clean"])
y_tr = train_df["label"].values
y_te = test_df["label"].values

print(f"Train matrix: {X_tr.shape} | Test matrix: {X_te.shape}")


## 3. Training & 5-Fold Stratified Cross-Validation


In [ ]:
# 1. Logistic Regression
lr_gs = GridSearchCV(LogisticRegression(max_iter=1000, random_state=SEED), {"C": [0.5, 1.0, 5.0]}, cv=5, scoring="f1_macro", n_jobs=-1)
lr_gs.fit(X_tr, y_tr)
best_lr = lr_gs.best_estimator_
print(f"LR Best Params: {lr_gs.best_params_} (CV F1: {lr_gs.best_score_:.4f})")

# 2. Linear SVM with Probability Calibration
svm_gs = GridSearchCV(LinearSVC(random_state=SEED, max_iter=2000), {"C": [0.1, 0.5, 1.0]}, cv=5, scoring="f1_macro", n_jobs=-1)
svm_gs.fit(X_tr, y_tr)
cal_svm = CalibratedClassifierCV(svm_gs.best_estimator_, cv=5).fit(X_tr, y_tr)
print(f"SVM Best Params: {svm_gs.best_params_} (CV F1: {svm_gs.best_score_:.4f})")

# 3. Multinomial Naive Bayes
nb_gs = GridSearchCV(MultinomialNB(), {"alpha": [0.1, 0.5, 1.0]}, cv=5, scoring="f1_macro", n_jobs=-1)
nb_gs.fit(X_tr, y_tr)
best_nb = nb_gs.best_estimator_
print(f"NB Best Params: {nb_gs.best_params_} (CV F1: {nb_gs.best_score_:.4f})")


## 4. Evaluation on Held-Out Test Set


In [ ]:
models = {"Logistic Regression": best_lr, "Linear SVM (Calibrated)": cal_svm, "Multinomial NB": best_nb}
results = []

for name, clf in models.items():
    pred = clf.predict(X_te)
    proba = clf.predict_proba(X_te)[:, 1] if hasattr(clf, "predict_proba") else None
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, average="macro"),
        "Recall": recall_score(y_te, pred, average="macro"),
        "F1-Macro": f1_score(y_te, pred, average="macro"),
        "ROC-AUC": roc_auc_score(y_te, proba) if proba is not None else None
    })

results_df = pd.DataFrame(results)
display(results_df)


## 5. Confusion Matrix (Logistic Regression)


In [ ]:
cm = confusion_matrix(y_te, best_lr.predict(X_te))
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Fake (0)", "Real (1)"], yticklabels=["Fake (0)", "Real (1)"], ax=ax)
ax.set_title("Confusion Matrix: Logistic Regression")
ax.set_ylabel("True Label")
ax.set_xlabel("Predicted Label")
plt.show()


## 6. Global Feature Explainability (Top Terms Driving Predictions)


In [ ]:
global_terms = get_global_influential_terms(vec, best_lr, top_n=20)
print("Top terms pushing towards FAKE (Negative Coefs):")
print([t["term"] for t in global_terms["fake_terms"][:10]])

print("\nTop terms pushing towards REAL (Positive Coefs):")
print([t["term"] for t in global_terms["real_terms"][:10]])
